<a href="https://colab.research.google.com/github/Sernoth/generative-models-cats/blob/main/generative_cats_notebook_wae_gan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project III: Generative Models — Cat Image Generation
FastGAN · PatchDiffusion · WAE-MMD · OT-CFM

Each model section runs a full **grid search** over the hyperparameters specified in the project plan.
Every (model, hparam-combo) gets its own Drive checkpoint, and results are collected in a shared
DataFrame at the end.

All default hyperparameter values are aligned with the original papers:
- FastGAN: `nz=256, lr=2e-4` (Liu et al. 2021, official `train.py`)
- PatchDiffusion: `σ_min=0.002, σ_max=80, P_mean=-1.2, P_std=1.2` (Karras et al. 2022, Table 1)
- WAE-MMD: `dz=64, λ=100, lr=1e-3` (Tolstikhin et al. 2019, official code)
- OT-CFM: `sigma=0.0, lr=2e-4` (Tong et al. 2024, torchcfm examples)

## 0. Google Drive · Install · Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/diffusion_cats'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive mounted →', DRIVE_DIR)

Mounted at /content/drive
Drive mounted → /content/drive/MyDrive/diffusion_cats


In [ ]:
!pip install -q torch torchvision diffusers accelerate datasets clean-fid torchmetrics opendatasets torch-fidelity torchcfm torchdiffeq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 144.2 MB/s eta 0:00:00


In [ ]:
import os, torch, numpy as np, itertools, json, pandas as pd
from pathlib import Path
from torchvision import transforms, utils
from torch.utils.data import DataLoader, Dataset
from torch import nn, optim
from tqdm.auto import tqdm
from PIL import Image
from cleanfid import fid as clean_fid
import os
import csv
import itertools
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from torchdiffeq import odeint
from torch.nn.utils import clip_grad_norm_
from diffusers import UNet2DModel
from pathlib import Path
from torchcfm.conditional_flow_matching import ExactOptimalTransportConditionalFlowMatcher
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE   = 64
BATCH      = 32
EPOCHS     = 50      # epochs per hparam combo
CKPT_EVERY = 10      # checkpoint cadence (epochs)
N_FAKE     = 100     # na 100 zmienic generated images used for FID/KID scoring per combo
OUT_DIR    = Path('generated'); OUT_DIR.mkdir(exist_ok=True)
REAL_DIR   = Path('real_eval'); REAL_DIR.mkdir(exist_ok=True)

RESULTS    = {}      # (model__slug) -> {FID, KID}
print(DEVICE)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


cuda


## Helpers: checkpointing + scoring

**Checkpoint filename convention:** `{tag}__{hparam_slug}__ep{epoch:03d}.pt`  
e.g. `fastgan__nz256_ngf64_lr2e-4__ep010.pt`

In [ ]:
# ── checkpoint I/O ────────────────────────────────────────────────────────────
def _ckpt_path(tag, slug, epoch):
    return Path(DRIVE_DIR) / f'{tag}__{slug}__ep{epoch:03d}.pt'

def save_ckpt(tag, slug, epoch, **state):
    p = _ckpt_path(tag, slug, epoch)
    torch.save({'epoch': epoch, **state}, p)
    print(f'  [ckpt] {p.name}')

def load_ckpt(tag, slug):
    matches = sorted(Path(DRIVE_DIR).glob(f'{tag}__{slug}__ep*.pt'))
    if not matches:
        print(f'  [ckpt] no checkpoint for {tag} ({slug}) — starting fresh'); return None, 0
    state = torch.load(matches[-1], map_location=DEVICE)
    epoch = state.pop('epoch')
    print(f'  [ckpt] resumed {matches[-1].name} (ep {epoch})'); return state, epoch

# ── FID / KID scoring ─────────────────────────────────────────────────────────
def score(gen_dir: str, tag: str, slug: str):
    key = f'{tag}__{slug}'
    RESULTS[key] = {
        'FID': round(clean_fid.compute_fid(str(REAL_DIR), gen_dir, verbose = False), 2),
        'KID': round(clean_fid.compute_kid(str(REAL_DIR), gen_dir, verbose = False), 4),
    }
    print(f'  [score] {key}: {RESULTS[key]}')
    return RESULTS[key]

def save_imgs(tensors, directory):
    Path(directory).mkdir(parents=True, exist_ok=True)
    existing = len(list(Path(directory).glob('*.png')))
    for i, t in enumerate(tensors):
        utils.save_image((t.clamp(-1,1)+1)/2, f'{directory}/{existing+i}.png')

def build_real_ref(loader, n=1000):
    if len(list(REAL_DIR.glob('*.png'))) >= n: return
    print('Building real reference set...')
    count = 0
    for imgs in loader:
        for img in imgs:
            utils.save_image((img+1)/2, REAL_DIR/f'{count}.png'); count += 1
            if count >= n: return

## 1. Data

In [ ]:
import opendatasets as od
od.download('https://www.kaggle.com/datasets/crawford/cat-dataset')
CAT_DIR = 'cat-dataset'


Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: apollosatori
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/crawford/cat-dataset


100%|██████████| 4.04G/4.04G [03:15<00:00, 22.2MB/s]


In [ ]:
tf = transforms.Compose([
    transforms.Resize(IMG_SIZE), transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize([.5]*3, [.5]*3),
])

class FlatImageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.files = list(Path(root).rglob('*.jpg')) + list(Path(root).rglob('*.png'))
        self.transform = transform
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert('RGB')
        return self.transform(img) if self.transform else img

dataset = FlatImageDataset(CAT_DIR, tf)
loader  = DataLoader(dataset, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
build_real_ref(loader)
CFM_BATCH  = 64
cfm_loader = DataLoader(dataset, batch_size=CFM_BATCH, shuffle=True,
                        num_workers=2, pin_memory=True)
print(f'{len(dataset)} training images, {len(list(REAL_DIR.glob("*.png")))} real reference images')

NameError: name 'CAT_DIR' is not defined

## 3. PatchDiffusion — grid search



In [ ]:
from diffusers import UNet2DModel, DDPMScheduler
CSV_OUTPUT_PATH = os.path.join(DRIVE_DIR, 'patchdiffusion_results.csv')
SIGMA_MIN, SIGMA_MAX = 0.002, 80.0
P_STD = 1.2  # fixed

DIFF_GRID = dict(
    patch_size = [16, 32],
    p_mean     = [-1.2, -0.6],
    lr         = [1e-4, 5e-5],
)
def heun_sample(unet, n_steps=35, batch_size=16):
    rho  = 7.0
    sigs = (SIGMA_MAX**(1/rho) + torch.linspace(0, 1, n_steps, device=DEVICE) *
            (SIGMA_MIN**(1/rho) - SIGMA_MAX**(1/rho))) ** rho
    sigs = torch.cat([sigs, sigs.new_zeros(1)])  # dopisanie 0 na końcu

    x = torch.randn(batch_size, 3, IMG_SIZE, IMG_SIZE, device=DEVICE) * sigs[0]
    unet.eval()
    with torch.no_grad():
        for i in range(len(sigs) - 1):
            s, s_next = sigs[i], sigs[i+1]
            t_idx = ((s / SIGMA_MAX) * 999).long().expand(batch_size)
            d     = (x - unet(x, t_idx).sample) / s          # kierunek score
            x_2   = x + (s_next - s) * d                     # krok Eulera
            if s_next > 0:
                t2  = ((s_next / SIGMA_MAX) * 999).long().expand(batch_size)
                d2  = (x_2 - unet(x_2, t2).sample) / s_next  # druga pochodna
                x   = x + (s_next - s) * (d + d2) / 2        # korekta Heuna
            else:
                x = x_2
    return x.clamp(-1, 1)


for patch_size, p_mean, lr in itertools.product(*DIFF_GRID.values()):
    slug = f'patch{patch_size}_pmean{p_mean}_lr{lr:.0e}'


    unet = UNet2DModel(
        sample_size=patch_size, in_channels=3, out_channels=3,
        layers_per_block=2, block_out_channels=(64, 128, 256),
        down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
        up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
    ).to(DEVICE)

    scheduler = DDPMScheduler(num_train_timesteps=1000, beta_schedule='squaredcos_cap_v2')
    opt       = optim.AdamW(unet.parameters(), lr=lr)

    state, start_ep = load_ckpt('patchdiff', slug)
    if state:
        unet.load_state_dict(state['unet'])
        opt.load_state_dict(state['opt'])
        print(f'[WCHODZENIE] Wznowiono trening od epoki: {start_ep}')

    for epoch in tqdm(range(start_ep, EPOCHS), desc=slug, leave=False):
        unet.train()
        for imgs in loader:
            imgs    = imgs.to(DEVICE)
            patches = transforms.RandomCrop(patch_size)(imgs)
            noise   = torch.randn_like(patches)

            # Log-normal noise level sampling (Karras et al. 2022)
            log_sig = torch.randn(patches.size(0), device=DEVICE) * P_STD + p_mean
            t       = (log_sig.exp().clamp(SIGMA_MIN, SIGMA_MAX) / SIGMA_MAX * 999).long()

            loss    = nn.functional.mse_loss(
                unet(scheduler.add_noise(patches, noise, t), t).sample, noise)

            opt.zero_grad()
            loss.backward()
            opt.step()

        current_epoch = epoch + 1
        if current_epoch % CKPT_EVERY == 0 or epoch == EPOCHS - 1:

            # A. Zapis wag modelu (.pt)
            save_ckpt('patchdiff', slug, current_epoch, unet=unet.state_dict(), opt=opt.state_dict())

            gen_dir = str(OUT_DIR / f'patchdiff_{slug}_ep{current_epoch:03d}')
            Path(gen_dir).mkdir(exist_ok=True)

            print(f'\n[Inference] generating {N_FAKE} images for epoch {current_epoch}...')
            for _ in range(0, N_FAKE, 16):
                save_imgs(heun_sample(unet, n_steps=35, batch_size=16).cpu(), gen_dir)

            metrics = score(gen_dir, 'patchdiff', slug)
            row = {
                'model_name': f'patchdiff_{slug}',
                'patch_size': patch_size,
                'p_mean': p_mean,
                'lr': lr,
                'epoch': current_epoch,
                'FID': metrics.get('FID') if metrics else None,
                'KID': metrics.get('KID') if metrics else None
            }

            if os.path.exists(CSV_OUTPUT_PATH):
                existing_df = pd.read_csv(CSV_OUTPUT_PATH)
                duplicate_mask = (existing_df['model_name'] == row['model_name']) & (existing_df['epoch'] == row['epoch'])
                existing_df = existing_df[~duplicate_mask]

                # Dołączamy nowy wiersz na koniec tabeli
                new_row_df = pd.DataFrame([row])
                updated_df = pd.concat([existing_df, new_row_df], ignore_index=True)
                updated_df.to_csv(CSV_OUTPUT_PATH, index=False)
            else:
                new_df = pd.DataFrame([row])
                new_df.to_csv(CSV_OUTPUT_PATH, index=False)

            print(f"  [save CSV] Wyniki modelu {row['model_name']} z epoki {current_epoch} zostały zsynchronizowane z Google Drive.")



Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.



=== URUCHAMIANIE: PatchDiffusion | patch16_pmean-1.2_lr1e-04 ===
  [ckpt] resumed patchdiff__patch16_pmean-1.2_lr1e-04__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening od epoki: 50


patch16_pmean-1.2_lr1e-04: 0it [00:00, ?it/s]


=== URUCHAMIANIE: PatchDiffusion | patch16_pmean-1.2_lr5e-05 ===
  [ckpt] resumed patchdiff__patch16_pmean-1.2_lr5e-05__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening od epoki: 50


patch16_pmean-1.2_lr5e-05: 0it [00:00, ?it/s]

...

patch32_pmean-1.2_lr5e-05: 0it [00:00, ?it/s]


=== URUCHAMIANIE: PatchDiffusion | patch32_pmean-0.6_lr1e-04 ===
  [ckpt] resumed patchdiff__patch32_pmean-0.6_lr1e-04__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening od epoki: 50


patch32_pmean-0.6_lr1e-04: 0it [00:00, ?it/s]


=== URUCHAMIANIE: PatchDiffusion | patch32_pmean-0.6_lr5e-05 ===
  [ckpt] resumed patchdiff__patch32_pmean-0.6_lr5e-05__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening od epoki: 50


patch32_pmean-0.6_lr5e-05: 0it [00:00, ?it/s]

## 4. WAE-MMD — grid search


In [ ]:
from torchvision.models import resnet18

WAE_GRID = dict(
    dz  = [64, 128],
    lam = [10.0, 100.0],
    lr  = [1e-3, 3e-4],
)

def mmd_rbf(x, y, sigma=1.0):
    """Unbiased RBF-kernel MMD (Tolstikhin et al. 2019, Eq. 4)."""
    def K(a, b): return torch.exp(-torch.cdist(a,b)**2 / (2*sigma**2))
    return K(x,x).mean() + K(y,y).mean() - 2*K(x,y).mean()

for dz, lam, lr in itertools.product(*WAE_GRID.values()):
    slug = f'dz{dz}_lam{lam:.0f}_lr{lr:.0e}'
    print(f'\n=== WAE-MMD | {slug} ===')

    class Encoder(nn.Module):
        def __init__(self):
            super().__init__()
            base = resnet18()
            self.net = nn.Sequential(*list(base.children())[:-1], nn.Flatten(), nn.Linear(512, dz))
        def forward(self, x): return self.net(x)

    class Decoder(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(dz, 512*4*4), nn.Unflatten(1,(512,4,4)),
                *[nn.Sequential(nn.Upsample(scale_factor=2), nn.Conv2d(ci,co,3,1,1), nn.ReLU())
                  for ci,co in [(512,256),(256,128),(128,64),(64,32)]],
                nn.Upsample(size=IMG_SIZE), nn.Conv2d(32,3,3,1,1), nn.Tanh(),
            )
        def forward(self, z): return self.net(z)

    enc, dec = Encoder().to(DEVICE), Decoder().to(DEVICE)
    opt = optim.Adam(list(enc.parameters())+list(dec.parameters()), lr=lr)

    state, start_ep = load_ckpt('waemmd', slug)
    if state:
        enc.load_state_dict(state['enc']); dec.load_state_dict(state['dec'])
        opt.load_state_dict(state['opt'])

    for epoch in tqdm(range(start_ep, EPOCHS), desc=slug, leave=False):
        for imgs in loader:
            imgs = imgs.to(DEVICE); z = enc(imgs)
            loss = nn.functional.mse_loss(dec(z), imgs) + lam * mmd_rbf(z, torch.randn_like(z))
            opt.zero_grad(); loss.backward(); opt.step()

        if (epoch+1) % CKPT_EVERY == 0 or epoch == EPOCHS-1:
            save_ckpt('waemmd', slug, epoch+1,
                      enc=enc.state_dict(), dec=dec.state_dict(), opt=opt.state_dict())

    dec.eval(); gen_dir = str(OUT_DIR/f'waemmd_{slug}')
    with torch.no_grad():
        for _ in range(0, N_FAKE, BATCH):
            save_imgs(dec(torch.randn(BATCH, dz, device=DEVICE)).cpu(), gen_dir)
    score(gen_dir, 'waemmd', slug)

## 5. OT-CFM — grid search


In [ ]:
CSV_OTCFM_PATH = os.path.join(DRIVE_DIR, 'ot_cfm_results.csv')

CFM_GRID = dict(
    sigma         = [0.0, 0.01],
    unet_channels = [
        (128, 256, 256, 256),
        (64,  128, 128, 128),
    ],
    lr            = [2e-4, 1e-4],
)

EMA_DECAY   = 0.9999
DROPOUT     = 0.1
GRAD_CLIP   = 1.0
CKPT_EVERY  = 10

def update_ema(ema_model, model, decay):
    """EMA update — ema_model stays on CPU, model is on GPU."""
    with torch.no_grad():
        for ema_p, p in zip(ema_model.parameters(), model.parameters()):
            ema_p.mul_(decay).add_(p.cpu(), alpha=1 - decay)

for sigma, unet_channels, lr in itertools.product(*CFM_GRID.values()):
    ch_str = '_'.join(str(c) for c in unet_channels)
    slug   = f'sig{sigma}_unet{ch_str}_lr{lr:.0e}'
    print(f'\n=== OT-CFM | {slug} ===')

    num_blocks  = len(unet_channels)
    down_blocks = ("DownBlock2D",) + ("AttnDownBlock2D",) * (num_blocks - 1)
    up_blocks   = ("AttnUpBlock2D",) * (num_blocks - 1) + ("UpBlock2D",)

    def make_unet():
        return UNet2DModel(
            sample_size=IMG_SIZE, in_channels=3, out_channels=3,
            layers_per_block=2, block_out_channels=unet_channels,
            down_block_types=down_blocks, up_block_types=up_blocks,
            dropout=DROPOUT,
        )

    fm_unet  = make_unet().to(DEVICE)

    ema_unet = make_unet().cpu()
    ema_unet.load_state_dict(fm_unet.state_dict())
    ema_unet.requires_grad_(False)

    cfm = ExactOptimalTransportConditionalFlowMatcher(sigma=sigma)
    opt = optim.Adam(fm_unet.parameters(), lr=lr, betas=(0.9, 0.999))

    state, start_ep = load_ckpt('otcfm', slug)
    if state:
        fm_unet.load_state_dict(state['unet'])
        ema_unet.load_state_dict(state['ema'])
        opt.load_state_dict(state['opt'])
        print(f'[WCHODZENIE] Wznowiono trening OT-CFM od epoki: {start_ep}')

    for epoch in tqdm(range(start_ep, EPOCHS), desc=slug, leave=False):
        fm_unet.train()
        for x1 in cfm_loader:
            x1 = x1.to(DEVICE)
            t, xt, ut = cfm.sample_location_and_conditional_flow(torch.randn_like(x1), x1)
            loss = nn.functional.mse_loss(fm_unet(xt, (t * 999).long()).sample, ut)
            opt.zero_grad()
            loss.backward()
            clip_grad_norm_(fm_unet.parameters(), GRAD_CLIP)
            opt.step()
            update_ema(ema_unet, fm_unet, EMA_DECAY)

        current_epoch = epoch + 1
        if current_epoch % CKPT_EVERY == 0 or epoch == EPOCHS - 1:

            # A. Zapis wag
            save_ckpt('otcfm', slug, current_epoch,
                      unet=fm_unet.state_dict(),
                      ema=ema_unet.state_dict(),  # CPU state_dict — torch.save obsługuje normalnie
                      opt=opt.state_dict())

            ema_unet.to(DEVICE).eval()
            gen_dir = str(OUT_DIR / f'otcfm_{slug}_ep{current_epoch:03d}')
            Path(gen_dir).mkdir(exist_ok=True)
            print(f'\n[Inference] Generowanie {N_FAKE} obrazków dla epoki {current_epoch}...')
            with torch.no_grad():
                for _ in range(0, N_FAKE, CFM_BATCH):
                    x0 = torch.randn(CFM_BATCH, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
                    def vf(t, x): return ema_unet(x, (t.expand(x.size(0)) * 999).long()).sample
                    samples = odeint(vf, x0, torch.linspace(0, 1, 20, device=DEVICE), method='euler')[-1]
                    save_imgs(samples.cpu(), gen_dir)
            ema_unet.cpu()
            torch.cuda.empty_cache()

            # C. Scoring + CSV
            metrics = score(gen_dir, 'otcfm', slug)
            row = {
                'model_name': f'otcfm_{slug}',
                'sigma':        sigma,
                'unet_channels': ch_str,
                'lr':           lr,
                'epoch':        current_epoch,
                'FID':          metrics.get('FID') if metrics else None,
                'KID':          metrics.get('KID') if metrics else None,
            }

            if os.path.exists(CSV_OTCFM_PATH):
                df_existing = pd.read_csv(CSV_OTCFM_PATH)
                mask = (df_existing['model_name'] == row['model_name']) & \
                       (df_existing['epoch']      == row['epoch'])
                df_existing = df_existing[~mask]
                pd.concat([df_existing, pd.DataFrame([row])], ignore_index=True) \
                  .to_csv(CSV_OTCFM_PATH, index=False)
            else:
                pd.DataFrame([row]).to_csv(CSV_OTCFM_PATH, index=False)

            print(f"  [ZAPIS CSV] {row['model_name']} ep{current_epoch} → {CSV_OTCFM_PATH}")

    # Zwolnij pamięć GPU po zakończeniu kombinacji
    del fm_unet, ema_unet, opt, cfm
    torch.cuda.empty_cache()


=== OT-CFM | sig0.0_unet128_256_256_256_lr2e-04 ===
  [ckpt] resumed otcfm__sig0.0_unet128_256_256_256_lr2e-04__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening OT-CFM od epoki: 50



=== OT-CFM | sig0.0_unet128_256_256_256_lr1e-04 ===


  [ckpt] resumed otcfm__sig0.0_unet128_256_256_256_lr1e-04__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening OT-CFM od epoki: 50



=== OT-CFM | sig0.0_unet64_128_128_128_lr2e-04 ===


  [ckpt] resumed otcfm__sig0.0_unet64_128_128_128_lr2e-04__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening OT-CFM od epoki: 50


...

  [ckpt] resumed otcfm__sig0.01_unet128_256_256_256_lr1e-04__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening OT-CFM od epoki: 50



=== OT-CFM | sig0.01_unet64_128_128_128_lr2e-04 ===


  [ckpt] resumed otcfm__sig0.01_unet64_128_128_128_lr2e-04__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening OT-CFM od epoki: 50



=== OT-CFM | sig0.01_unet64_128_128_128_lr1e-04 ===


  [ckpt] resumed otcfm__sig0.01_unet64_128_128_128_lr1e-04__ep050.pt (ep 50)
[WCHODZENIE] Wznowiono trening OT-CFM od epoki: 50


In [ ]:
# 6. Full training

In [ ]:
# WAE - MMD

In [ ]:

DZ             = 64
LAM            = 100
BATCH_WAE      = 100
LR_WAE         = 1e-3
EPOCHS_WAE     = 1200
CKPT_EVERY_WAE = 50
SLUG_WAE       = f'FULL_dz{DZ}_lam{LAM}_v3'   # v3 — nowy slug żeby nie mieszać z poprzednimi
num_workers = os.cpu_count() - 1 # lub os.cpu_count() - 1
wae_loader = DataLoader(dataset, batch_size=BATCH_WAE, shuffle=True,
                        num_workers=num_workers, pin_memory=True)

# ── architektura (Tolstikhin et al. 2019, Appendix C.2) ──────────────
class WAEEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,   128, 5, 2, 2), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 5, 2, 2), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 512, 5, 2, 2), nn.BatchNorm2d(512), nn.ReLU(),
            nn.Conv2d(512,1024, 5, 2, 2), nn.BatchNorm2d(1024),nn.ReLU(),
            nn.Flatten(),
            nn.Linear(1024 * 4 * 4, DZ),
        )
    def forward(self, x): return self.net(x)

class WAEDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc  = nn.Linear(DZ, 1024 * 8 * 8)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(1024, 512, 5, 2, 2, 1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.ConvTranspose2d(512,  256, 5, 2, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.ConvTranspose2d(256,  128, 5, 2, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128,    3, 5, 1, 2),    nn.Tanh(),
        )
    def forward(self, z): return self.net(self.fc(z).view(-1, 1024, 8, 8))

# ── multi-scale RBF MMD (oficjalny kod WAE, nie single-kernel) ────────
# Single sigma=1.0 jest martwe przy DZ=64 — odległości w przestrzeni
# latentnej są rzędu 10-100, kernel zwraca ~0 dla wszystkich par
def mmd_rbf(x, y, sigmas=(1.0, 2.0, 5.0, 10.0, 20.0, 50.0)):
    """Multi-scale RBF MMD — Tolstikhin et al. 2019, oficjalny kod."""
    def K(a, b, s):
        return torch.exp(-torch.cdist(a, b)**2 / (2 * s**2))
    result = sum(K(x,x,s).mean() + K(y,y,s).mean() - 2*K(x,y,s).mean()
                 for s in sigmas)
    return result / len(sigmas)

# ── LR schedule z paperu przeliczony na 9k obrazów ───────────────────
def lr_lambda(epoch):
    if epoch >= 1000: return 1/(2*5)   # 1e-4
    if epoch >= 600:  return 1/2       # 5e-4
    return 1.0                         # 1e-3

# ── inicjalizacja modelu ──────────────────────────────────────────────
enc = WAEEncoder().to(DEVICE)
dec = WAEDecoder().to(DEVICE)
opt_wae = optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=LR_WAE)

FINAL_WAE_BASE = Path(DRIVE_DIR) / 'final_wae_v3'
FINAL_WAE_BASE.mkdir(parents=True, exist_ok=True)
csv_path = FINAL_WAE_BASE / 'metrics_history.csv'

# ── resume z poprawnym last_epoch ─────────────────────────────────────
# POPRAWKA: last_epoch=start_ep-1 żeby scheduler wiedział gdzie jest po wznowieniu
# Bez tego scheduler resetuje się do ep0 i LR jest błędny
state, start_ep = load_ckpt('waemmd', SLUG_WAE)
if state:
    enc.load_state_dict(state['enc'])
    dec.load_state_dict(state['dec'])
    opt_wae.load_state_dict(state['opt'])
    # NIE wczytujemy state['sched'] — budujemy nowy z poprawnym last_epoch
    scheduler_wae = LambdaLR(opt_wae, lr_lambda, last_epoch=start_ep - 1)
    print(f'[resume] od epoki {start_ep} | lr={scheduler_wae.get_last_lr()[0]:.2e}')
else:
    # Świeży trening — last_epoch=-1 (domyślne)
    scheduler_wae = LambdaLR(opt_wae, lr_lambda)

# ── trening ───────────────────────────────────────────────────────────
for epoch in tqdm(range(start_ep, EPOCHS_WAE), desc='WAE-MMD FULL v3'):
    enc.train(); dec.train()
    epoch_loss = epoch_recon = epoch_mmd = 0.0

    for imgs in wae_loader:
        imgs  = imgs.to(DEVICE)
        z     = enc(imgs)
        recon = dec(z)

        recon_loss = nn.functional.mse_loss(recon, imgs)
        mmd_loss   = mmd_rbf(z, torch.randn_like(z))
        loss       = recon_loss + LAM * mmd_loss

        opt_wae.zero_grad(); loss.backward(); opt_wae.step()
        epoch_loss  += loss.item()
        epoch_recon += recon_loss.item()
        epoch_mmd   += mmd_loss.item()

    scheduler_wae.step()

    current_epoch = epoch + 1
    if current_epoch % CKPT_EVERY_WAE == 0 or epoch == EPOCHS_WAE - 1:

        # A. Checkpoint
        save_ckpt('waemmd', SLUG_WAE, current_epoch,
                  enc=enc.state_dict(), dec=dec.state_dict(),
                  opt=opt_wae.state_dict())
        # Uwaga: sched NIE jest zapisywany — odbudowujemy go z last_epoch przy resume

        # B. Generowanie + FID
        enc.eval(); dec.eval()
        epoch_dir = FINAL_WAE_BASE / f'eval_ep{current_epoch:04d}'
        epoch_dir.mkdir(exist_ok=True)
        with torch.no_grad():
            for _ in range(0, 1000, BATCH_WAE):
                save_imgs(dec(torch.randn(BATCH_WAE, DZ, device=DEVICE)).cpu(),
                          str(epoch_dir))

        score(str(epoch_dir), 'waemmd', f'{SLUG_WAE}_ep{current_epoch:04d}')
        key = f'waemmd__{SLUG_WAE}_ep{current_epoch:04d}'
        fid = RESULTS.get(key, {}).get('FID', 'N/A')
        kid = RESULTS.get(key, {}).get('KID', 'N/A')

        n_batches  = len(wae_loader)
        avg_loss   = epoch_loss  / n_batches
        avg_recon  = epoch_recon / n_batches
        avg_mmd    = epoch_mmd   / n_batches
        current_lr = scheduler_wae.get_last_lr()[0]

        # C. CSV — logujemy też recon i mmd osobno żeby wykryć czy MMD działa
        file_exists = csv_path.exists()
        with open(csv_path, 'a', newline='') as f:
            w = csv.writer(f)
            if not file_exists:
                w.writerow(['epoch', 'imgs_seen_M', 'loss',
                            'recon_loss', 'mmd_loss', 'FID', 'KID', 'lr'])
            w.writerow([
                current_epoch,
                round(current_epoch * len(dataset) / 1e6, 2),
                round(avg_loss,  4),
                round(avg_recon, 4),
                round(avg_mmd,   6),   # MMD powinno być >0 — jeśli ~0 to kernel martwy
                fid, kid,
                f'{current_lr:.2e}',
            ])
        print(f'  ep{current_epoch} | loss={avg_loss:.4f} '
              f'recon={avg_recon:.4f} mmd={avg_mmd:.6f} '
              f'| FID={fid} | lr={current_lr:.2e}')

        enc.train(); dec.train()
        torch.cuda.empty_cache()

# ── interpolacja latentna ─────────────────────────────────────────────
enc.eval(); dec.eval()
imgs_sample = next(iter(wae_loader))[:2].to(DEVICE)
with torch.no_grad():
    z1, z2 = enc(imgs_sample[0:1]), enc(imgs_sample[1:2])
    torch.save(z1.cpu(), Path(DRIVE_DIR)/'wae_z1.pt')
    torch.save(z2.cpu(), Path(DRIVE_DIR)/'wae_z2.pt')
    interp = [dec((1 - i/9)*z1 + (i/9)*z2) for i in range(10)]
    utils.save_image(
        utils.make_grid(torch.cat(interp).clamp(-1,1), nrow=10, normalize=True),
        FINAL_WAE_BASE/'interpolation.png'
    )
    print('Interpolacja →', FINAL_WAE_BASE/'interpolation.png')

  [ckpt] resumed waemmd__FULL_dz64_lam100_v3__ep650.pt (ep 650)
[resume] od epoki 650 | lr=5.00e-04


WAE-MMD FULL v3:   0%|          | 0/550 [00:00<?, ?it/s]

  [ckpt] waemmd__FULL_dz64_lam100_v3__ep700.pt
  [score] waemmd__FULL_dz64_lam100_v3_ep0700: {'FID': np.float64(308.2), 'KID': 0.3433}
  ep700 | loss=2.2407 recon=0.0391 mmd=0.022016 | FID=308.2 | lr=5.00e-04
  [ckpt] waemmd__FULL_dz64_lam100_v3__ep750.pt
  [score] waemmd__FULL_dz64_lam100_v3_ep0750: {'FID': np.float64(311.11), 'KID': 0.3495}
  ep750 | loss=2.2826 recon=0.0430 mmd=0.022397 | FID=311.11 | lr=5.00e-04
  [ckpt] waemmd__FULL_dz64_lam100_v3__ep800.pt
  [score] waemmd__FULL_dz64_lam100_v3_ep0800: {'FID': np.float64(311.72), 'KID': 0.3502}
  ep800 | loss=2.2817 recon=0.0417 mmd=0.022400 | FID=311.72 | lr=5.00e-04
  [ckpt] waemmd__FULL_dz64_lam100_v3__ep850.pt
  [score] waemmd__FULL_dz64_lam100_v3_ep0850: {'FID': np.float64(308.13), 'KID': 0.3437}
  ep850 | loss=2.3133 recon=0.0374 mmd=0.022760 | FID=308.13 | lr=5.00e-04
  [ckpt] waemmd__FULL_dz64_lam100_v3__ep900.pt
  [score] waemmd__FULL_dz64_lam100_v3_ep0900: {'FID': np.float64(304.07), 'KID': 0.3379}
  ep900 | loss=2.2775 

In [ ]:
# OT-CFM full training

In [ ]:

SIGMA          = 1e-6
BATCH_CFM      = 64
LR_MAX         = 5e-4
GRAD_CLIP      = 1.0
EMA_DECAY      = 0.9999
DROPOUT        = 0.0
UNET_CH        = (64, 128, 128, 128)
EPOCHS_CFM     = 200
WARMUP_EP      = 10
CKPT_EVERY_CFM = 5
N_STEPS_EVAL   = 20    # szybsza inferencja podczas ewaluacji
N_STEPS_FINAL  = 100   # więcej kroków tylko dla finalnej interpolacji

SLUG_CFM       = f'FULL_sig{SIGMA}_unet{"_".join(str(c) for c in UNET_CH)}'
FINAL_CFM_BASE = Path(DRIVE_DIR) / 'final_otcfm'
FINAL_CFM_BASE.mkdir(parents=True, exist_ok=True)
csv_path_cfm   = FINAL_CFM_BASE / 'metrics_history.csv'

cfm_loader = DataLoader(dataset, batch_size=BATCH_CFM, shuffle=True,
                        num_workers=2, pin_memory=True)

# ── model ─────────────────────────────────────────────────────────────
num_blocks  = len(UNET_CH)
down_blocks = ("DownBlock2D",) + ("AttnDownBlock2D",) * (num_blocks - 1)
up_blocks   = ("AttnUpBlock2D",) * (num_blocks - 1) + ("UpBlock2D",)

fm_unet = UNet2DModel(
    sample_size=IMG_SIZE, in_channels=3, out_channels=3,
    layers_per_block=2, block_out_channels=UNET_CH,
    down_block_types=down_blocks, up_block_types=up_blocks,
    dropout=DROPOUT,
).to(DEVICE)

ema_unet = UNet2DModel(
    sample_size=IMG_SIZE, in_channels=3, out_channels=3,
    layers_per_block=2, block_out_channels=UNET_CH,
    down_block_types=down_blocks, up_block_types=up_blocks,
    dropout=DROPOUT,
).cpu()
ema_unet.load_state_dict(fm_unet.state_dict())
ema_unet.requires_grad_(False)

cfm     = ExactOptimalTransportConditionalFlowMatcher(sigma=SIGMA)
opt_cfm = optim.Adam(fm_unet.parameters(), lr=LR_MAX,
                     betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0)

def lr_lambda_cfm(epoch):
    if epoch < WARMUP_EP:
        return epoch / max(1, WARMUP_EP)
    progress = (epoch - WARMUP_EP) / max(1, EPOCHS_CFM - WARMUP_EP)
    return max(0.0, 1.0 - progress)

scheduler_cfm = torch.optim.lr_scheduler.LambdaLR(opt_cfm, lr_lambda_cfm)

def update_ema(ema_model, model, decay):
    with torch.no_grad():
        for ep, p in zip(ema_model.parameters(), model.parameters()):
            ep.mul_(decay).add_(p.cpu(), alpha=1 - decay)

def sample_from_z(model, z, n_steps=20):
    """Generuje obrazy startując od konkretnego tensora szumu z."""
    model.eval()
    with torch.no_grad():
        z = z.to(DEVICE)
        def vf(t, x):
            return model(x, (t.expand(x.size(0)) * 999).long()).sample
        traj = odeint(vf, z, torch.linspace(0, 1, n_steps, device=DEVICE),
                      method='dopri5', atol=1e-5, rtol=1e-5)
    return traj[-1].clamp(-1, 1)

# ── resume ────────────────────────────────────────────────────────────
state, start_ep = load_ckpt('otcfm', SLUG_CFM)
if state:
    fm_unet.load_state_dict(state['unet'])
    ema_unet.load_state_dict(state['ema'])
    opt_cfm.load_state_dict(state['opt'])
    scheduler_cfm.load_state_dict(state['sched'])
    print(f'[resume] od epoki {start_ep}')

# ── trening ───────────────────────────────────────────────────────────
for epoch in tqdm(range(start_ep, EPOCHS_CFM), desc='OT-CFM FULL'):
    fm_unet.train()
    epoch_loss = 0.0

    for x1 in cfm_loader:
        x1 = x1.to(DEVICE)
        t, xt, ut = cfm.sample_location_and_conditional_flow(torch.randn_like(x1), x1)
        loss = nn.functional.mse_loss(fm_unet(xt, (t * 999).long()).sample, ut)
        opt_cfm.zero_grad(); loss.backward()
        clip_grad_norm_(fm_unet.parameters(), GRAD_CLIP)
        opt_cfm.step()
        update_ema(ema_unet, fm_unet, EMA_DECAY)
        epoch_loss += loss.item()

    scheduler_cfm.step()

    current_epoch = epoch + 1
    if current_epoch % CKPT_EVERY_CFM == 0 or epoch == EPOCHS_CFM - 1:

        # A. Checkpoint
        save_ckpt('otcfm', SLUG_CFM, current_epoch,
                  unet=fm_unet.state_dict(), ema=ema_unet.state_dict(),
                  opt=opt_cfm.state_dict(), sched=scheduler_cfm.state_dict())

        # B. Generowanie przez EMA — n_steps=20 dla szybkości
        ema_unet.to(DEVICE).eval()
        epoch_dir = FINAL_CFM_BASE / f'eval_ep{current_epoch:04d}'
        epoch_dir.mkdir(exist_ok=True)
        print(f'\n[Inference] ep{current_epoch} — generowanie 1000 obrazów...')
        with torch.no_grad():
            for _ in range(0, 1000, BATCH_CFM):
                z = torch.randn(BATCH_CFM, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
                save_imgs(sample_from_z(ema_unet, z, n_steps=N_STEPS_EVAL).cpu(),
                          str(epoch_dir))
        ema_unet.cpu(); torch.cuda.empty_cache()

        # C. FID/KID
        score(str(epoch_dir), 'otcfm', f'{SLUG_CFM}_ep{current_epoch:04d}')
        key = f'otcfm__{SLUG_CFM}_ep{current_epoch:04d}'
        fid = RESULTS.get(key, {}).get('FID', 'N/A')
        kid = RESULTS.get(key, {}).get('KID', 'N/A')
        avg_loss   = epoch_loss / len(cfm_loader)
        current_lr = scheduler_cfm.get_last_lr()[0]

        # D. CSV
        file_exists = csv_path_cfm.exists()
        with open(csv_path_cfm, 'a', newline='') as f:
            w = csv.writer(f)
            if not file_exists:
                w.writerow(['epoch', 'imgs_seen_M', 'loss', 'FID', 'KID', 'lr'])
            w.writerow([current_epoch,
                        round(current_epoch * len(dataset) / 1e6, 3),
                        round(avg_loss, 5), fid, kid, f'{current_lr:.2e}'])
        print(f'  ep{current_epoch} | loss={avg_loss:.5f} | FID={fid} | lr={current_lr:.2e}')
        fm_unet.train()

# ── interpolacja latentna (n_steps=100 dla jakości) ───────────────────
ema_unet.to(DEVICE).eval()
INTERP_DIR = FINAL_CFM_BASE / 'interpolation'
INTERP_DIR.mkdir(exist_ok=True)

torch.manual_seed(0)
z1 = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
z2 = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
torch.save(z1, Path(DRIVE_DIR) / 'cfm_z1.pt')
torch.save(z2, Path(DRIVE_DIR) / 'cfm_z2.pt')

with torch.no_grad():
    interp_imgs = [sample_from_z(ema_unet, (1-i/9)*z1 + (i/9)*z2,
                                 n_steps=N_STEPS_FINAL)
                   for i in range(10)]
    utils.save_image(
        utils.make_grid(torch.cat(interp_imgs).clamp(-1,1), nrow=10, normalize=True),
        INTERP_DIR / 'interpolation.png')
    utils.save_image((interp_imgs[0]+1)/2,  INTERP_DIR/'img1.png')
    utils.save_image((interp_imgs[-1]+1)/2, INTERP_DIR/'img2.png')

ema_unet.cpu()
print('Interpolacja →', INTERP_DIR/'interpolation.png')

  [ckpt] resumed otcfm__FULL_sig1e-06_unet64_128_128_128__ep030.pt (ep 30)
[resume] od epoki 30


OT-CFM FULL:   0%|          | 0/170 [00:00<?, ?it/s]

  [ckpt] otcfm__FULL_sig1e-06_unet64_128_128_128__ep035.pt

[Inference] ep35 — generowanie 1000 obrazów...
  [score] otcfm__FULL_sig1e-06_unet64_128_128_128_ep0035: {'FID': np.float64(369.87), 'KID': 0.4438}
  ep35 | loss=0.12667 | FID=369.87 | lr=4.34e-04
  [ckpt] otcfm__FULL_sig1e-06_unet64_128_128_128__ep040.pt

[Inference] ep40 — generowanie 1000 obrazów...
  [score] otcfm__FULL_sig1e-06_unet64_128_128_128_ep0040: {'FID': np.float64(362.71), 'KID': 0.4348}
  ep40 | loss=0.12682 | FID=362.71 | lr=4.21e-04
  [ckpt] otcfm__FULL_sig1e-06_unet64_128_128_128__ep045.pt

[Inference] ep45 — generowanie 1000 obrazów...
  [score] otcfm__FULL_sig1e-06_unet64_128_128_128_ep0045: {'FID': np.float64(357.77), 'KID': 0.4247}
  ep45 | loss=0.12518 | FID=357.77 | lr=4.08e-04


KeyboardInterrupt: 

## 7. Experiment: Cats + Dogs

In [ ]:
import opendatasets as od

od.download('https://www.kaggle.com/competitions/dogs-vs-cats')

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: apollosatori
Your Kaggle Key: ··········


100%|██████████| 812M/812M [00:38<00:00, 21.9MB/s]



Extracting archive ./dogs-vs-cats/dogs-vs-cats.zip to ./dogs-vs-cats


In [ ]:
import zipfile
import os

with zipfile.ZipFile('./dogs-vs-cats/train.zip', 'r') as zip_ref:
    zip_ref.extractall('./dogs-vs-cats')

print(f"Number of images found: {len(os.listdir('./dogs-vs-cats/train'))}")

Number of images found: 25000


In [ ]:
tf = transforms.Compose([
    transforms.Resize(IMG_SIZE), transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize([.5]*3, [.5]*3),
])

class FlatImageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.files = list(Path(root).rglob('*.jpg')) + list(Path(root).rglob('*.png'))
        self.transform = transform
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        img = Image.open(self.files[i]).convert('RGB')
        return self.transform(img) if self.transform else img
dataset = FlatImageDataset('dogs-vs-cats/train', tf)
catdog_loader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=2)
print(f'Cat+Dog dataset: {len(dataset)} images')

Cat+Dog dataset: 25000 images


In [ ]:

NZ_V3          = 100
NGF_V3         = 64
NDF_V3         = 64
LR_V3          = 2e-4
BETAS_V3       = (0.5, 0.999)
BATCH_V3       = 128

SLUG_V3        = f'FULL_dcgan_v3_nz{NZ_V3}_ngf{NGF_V3}'
FINAL_V3_BASE  = Path(DRIVE_DIR) / 'final_dcgan_v3'
FINAL_V3_BASE.mkdir(parents=True, exist_ok=True)
csv_path_v3    = FINAL_V3_BASE / 'metrics_history.csv'

REAL_DIR = FINAL_V3_BASE / 'real_eval'
REAL_DIR.mkdir(parents=True, exist_ok=True)

v3_loader = DataLoader(
    dataset,
    batch_size=BATCH_V3,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

class DCGeneratorV3(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(NZ_V3, NGF_V3*8, 4, 1, 0),
            nn.BatchNorm2d(NGF_V3*8),
            nn.ReLU(),

            nn.ConvTranspose2d(NGF_V3*8, NGF_V3*4, 4, 2, 1),
            nn.BatchNorm2d(NGF_V3*4),
            nn.ReLU(),

            nn.ConvTranspose2d(NGF_V3*4, NGF_V3*2, 4, 2, 1),
            nn.BatchNorm2d(NGF_V3*2),
            nn.ReLU(),

            nn.ConvTranspose2d(NGF_V3*2, NGF_V3, 4, 2, 1),
            nn.BatchNorm2d(NGF_V3),
            nn.ReLU(),

            nn.ConvTranspose2d(NGF_V3, 3, 4, 2, 1),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z.view(-1, NZ_V3, 1, 1))

class DCDiscriminatorV3(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, NDF_V3, 4, 2, 1),
            nn.LeakyReLU(0.2),

            nn.Conv2d(NDF_V3, NDF_V3*2, 4, 2, 1),
            nn.BatchNorm2d(NDF_V3*2),
            nn.LeakyReLU(0.2),

            nn.Conv2d(NDF_V3*2, NDF_V3*4, 4, 2, 1),
            nn.BatchNorm2d(NDF_V3*4),
            nn.LeakyReLU(0.2),

            nn.Conv2d(NDF_V3*4, NDF_V3*8, 4, 2, 1),
            nn.BatchNorm2d(NDF_V3*8),
            nn.LeakyReLU(0.2),

            nn.Conv2d(NDF_V3*8, 1, 4, 1, 0),
        )

    def forward(self, x):
        return self.net(x).view(-1, 1)

def weights_init_v3(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

G_v3 = DCGeneratorV3().to(DEVICE)
G_v3.apply(weights_init_v3)

D_v3 = DCDiscriminatorV3().to(DEVICE)
D_v3.apply(weights_init_v3)

opt_G_v3 = optim.Adam(G_v3.parameters(), lr=LR_V3, betas=BETAS_V3)
opt_D_v3 = optim.Adam(D_v3.parameters(), lr=LR_V3, betas=BETAS_V3)

bce = nn.BCEWithLogitsLoss()

# ── MODE COLLAPSE DETECTION ───────────────────────────────────────────
def sample_diversity_v3(G, nz, n=64, device='cuda'):
    with torch.no_grad():
        z = torch.randn(n, nz, device=device)
        imgs = G(z)
    return imgs.std(dim=0).mean().item()

def is_mode_collapsed_v3(loss_D, loss_G, diversity, ep):
    if diversity < 0.05:
        return True, f'diversity={diversity:.4f} < 0.05'
    if loss_D < 0.01:
        return True, f'loss_D={loss_D:.4f} < 0.01 (D dominates)'
    if loss_G > 10.0:
        return True, f'loss_G={loss_G:.4f} > 10 (G cant cheat D)'
    return False, None

fixed_z_v3 = torch.randn(16, NZ_V3, device=DEVICE)

EPOCHS_V3     = 1500
CKPT_EVERY_V3 = 20

def count_images(directory):
    directory = Path(directory)
    exts = ("*.png", "*.jpg", "*.jpeg", "*.webp")
    return sum(len(list(directory.glob(ext))) for ext in exts)

def save_imgs(tensors, directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    existing = count_images(directory)

    for i, t in enumerate(tensors):
        out = directory / f'{existing + i:06d}.png'
        utils.save_image((t.clamp(-1, 1) + 1) / 2, out)

def build_real_ref(loader, n=1000, overwrite=False):
    REAL_DIR.mkdir(parents=True, exist_ok=True)

    if overwrite:
        for p in REAL_DIR.glob("*"):
            if p.is_file():
                p.unlink()

    current = count_images(REAL_DIR)
    if current >= n:
        print(f'[real] reference already exists: {current} images in {REAL_DIR}')
        return

    print('Building real reference set...')
    count = current
    for imgs in loader:
        for img in imgs:
            out = REAL_DIR / f'{count:06d}.png'
            utils.save_image((img.clamp(-1, 1) + 1) / 2, out)
            count += 1
            if count >= n:
                print(f'[real] saved {count} images to {REAL_DIR}')
                return

build_real_ref(v3_loader, n=1000)


state, start_ep = load_ckpt('dcgan_v3', SLUG_V3)
if state:
    G_v3.load_state_dict(state['G'])
    D_v3.load_state_dict(state['D'])
    opt_G_v3.load_state_dict(state['opt_G'])
    opt_D_v3.load_state_dict(state['opt_D'])
    print(f'[resume] od epoki {start_ep}')
    force_eval_on_start = True
else:
    force_eval_on_start = False

# ── TRENING V3 ────────────────────────────────────────────────────────
collapse_counter = 0

for epoch in tqdm(range(start_ep, EPOCHS_V3), desc='DCGAN v3'):
    G_v3.train()
    D_v3.train()
    epoch_loss_G = 0.0
    epoch_loss_D = 0.0

    for real in v3_loader:
        real = real.to(DEVICE)
        bs = real.size(0)
        z = torch.randn(bs, NZ_V3, device=DEVICE)

        # Discriminator
        fake = G_v3(z).detach()
        loss_D = bce(D_v3(real), torch.ones(bs, 1, device=DEVICE)) + \
                 bce(D_v3(fake), torch.zeros(bs, 1, device=DEVICE))
        opt_D_v3.zero_grad()
        loss_D.backward()
        opt_D_v3.step()

        # Generator
        loss_G = bce(D_v3(G_v3(z)), torch.ones(bs, 1, device=DEVICE))
        opt_G_v3.zero_grad()
        loss_G.backward()
        opt_G_v3.step()

        epoch_loss_G += loss_G.item()
        epoch_loss_D += loss_D.item()

    current_epoch = epoch + 1
    n = len(v3_loader)
    avg_G = epoch_loss_G / n
    avg_D = epoch_loss_D / n
    diversity = sample_diversity_v3(G_v3, NZ_V3, device=DEVICE)

    # ── mode collapse check ───────────────────────────────────────────
    collapsed, reason = is_mode_collapsed_v3(avg_D, avg_G, diversity, current_epoch)
    if collapsed:
        collapse_counter += 1
        print(f'    ep{current_epoch} MODE COLLAPSE suspected: {reason} (counter={collapse_counter}/5)')
        if collapse_counter >= 5:
            print('  Mode collapse confirmed by 5 epochs - stop of training.')
            save_ckpt(
                'dcgan_v3', SLUG_V3, current_epoch,
                G=G_v3.state_dict(),
                D=D_v3.state_dict(),
                opt_G=opt_G_v3.state_dict(),
                opt_D=opt_D_v3.state_dict()
            )
            break
    else:
        collapse_counter = 0

    if current_epoch % CKPT_EVERY_V3 == 0 or epoch == EPOCHS_V3 - 1 or (force_eval_on_start and epoch == start_ep):
        force_eval_on_start = False

        # A. Checkpoint
        save_ckpt(
            'dcgan_v3', SLUG_V3, current_epoch,
            G=G_v3.state_dict(),
            D=D_v3.state_dict(),
            opt_G=opt_G_v3.state_dict(),
            opt_D=opt_D_v3.state_dict()
        )

        # B. Generowanie
        G_v3.eval()
        epoch_dir = FINAL_V3_BASE / f'eval_ep{current_epoch:04d}'
        if epoch_dir.exists():
            import shutil
            shutil.rmtree(epoch_dir)
        epoch_dir.mkdir(parents=True, exist_ok=True)

        with torch.no_grad():
            fixed_imgs = G_v3(fixed_z_v3)
            utils.save_image(
                (fixed_imgs + 1) / 2,
                FINAL_V3_BASE / f'progress_ep{current_epoch:04d}.png',
                nrow=4
            )

            for _ in range(0, 1000, BATCH_V3):
                z_s = torch.randn(BATCH_V3, NZ_V3, device=DEVICE)
                save_imgs(G_v3(z_s).cpu(), str(epoch_dir))

        # C. FID/KID Scoring
        score(str(epoch_dir), 'dcgan_v3', f'{SLUG_V3}_ep{current_epoch:04d}')
        key = f'dcgan_v3__{SLUG_V3}_ep{current_epoch:04d}'
        fid = RESULTS.get(key, {}).get('FID', 'N/A')
        kid = RESULTS.get(key, {}).get('KID', 'N/A')

        # D. CSV Saving
        file_exists = csv_path_v3.exists()
        with open(csv_path_v3, 'a', newline='') as f:
            w = csv.writer(f)
            if not file_exists:
                w.writerow(['epoch', 'imgs_seen_M', 'loss_G', 'loss_D', 'diversity', 'FID', 'KID'])
            w.writerow([
                current_epoch,
                round(current_epoch * len(dataset) / 1e6, 3),
                round(avg_G, 4),
                round(avg_D, 4),
                round(diversity, 4),
                fid,
                kid
            ])

        print(f'  ep{current_epoch} | G={avg_G:.4f} D={avg_D:.4f} div={diversity:.4f} | FID={fid}')
        G_v3.train()
        D_v3.train()
        torch.cuda.empty_cache()

# interpolation latent
G_v3.eval()
INTERP_V3 = FINAL_V3_BASE / 'interpolation'
INTERP_V3.mkdir(exist_ok=True)

torch.manual_seed(42)
z1_v3 = torch.randn(1, NZ_V3)
z2_v3 = torch.randn(1, NZ_V3)
torch.save(z1_v3, Path(DRIVE_DIR) / 'dcgan_v3_z1.pt')
torch.save(z2_v3, Path(DRIVE_DIR) / 'dcgan_v3_z2.pt')

with torch.no_grad():
    interp_v3 = [G_v3(((1 - i / 9) * z1_v3 + (i / 9) * z2_v3).to(DEVICE)) for i in range(10)]
    utils.save_image(
        utils.make_grid(torch.cat(interp_v3).clamp(-1, 1), nrow=10, normalize=True),
        INTERP_V3 / 'interpolation.png'
    )
    utils.save_image((interp_v3[0].clamp(-1, 1) + 1) / 2, INTERP_V3 / 'img1.png')
    utils.save_image((interp_v3[-1].clamp(-1, 1) + 1) / 2, INTERP_V3 / 'img2.png')

print('Interpolacja v3 finished! path →', INTERP_V3 / 'interpolation.png')

[real] reference already exists: 1000 images in /content/drive/MyDrive/diffusion_cats/final_dcgan_v3/real_eval
  [ckpt] resumed dcgan_v3__FULL_dcgan_v3_nz100_ngf64__ep020.pt (ep 20)
[resume] od epoki 20


DCGAN v3:   0%|          | 0/1480 [00:00<?, ?it/s]

  [ckpt] dcgan_v3__FULL_dcgan_v3_nz100_ngf64__ep021.pt
[score] real=1000 gen=1024
  [score] dcgan_v3__FULL_dcgan_v3_nz100_ngf64_ep0021: {'FID': np.float64(208.64), 'KID': 0.1468}
  ep21 | G=2.9550 D=0.6520 div=0.4881 | FID=208.64
  [ckpt] dcgan_v3__FULL_dcgan_v3_nz100_ngf64__ep040.pt
[score] real=1000 gen=1024
  [score] dcgan_v3__FULL_dcgan_v3_nz100_ngf64_ep0040: {'FID': np.float64(185.91), 'KID': 0.126}
  ep40 | G=2.8787 D=0.5822 div=0.4934 | FID=185.91
  [ckpt] dcgan_v3__FULL_dcgan_v3_nz100_ngf64__ep060.pt
[score] real=1000 gen=1024
  [score] dcgan_v3__FULL_dcgan_v3_nz100_ngf64_ep0060: {'FID': np.float64(174.97), 'KID': 0.115}
  ep60 | G=2.9363 D=0.3922 div=0.4958 | FID=174.97
  [ckpt] dcgan_v3__FULL_dcgan_v3_nz100_ngf64__ep080.pt
[score] real=1000 gen=1024
  [score] dcgan_v3__FULL_dcgan_v3_nz100_ngf64_ep0080: {'FID': np.float64(172.49), 'KID': 0.1119}
  ep80 | G=3.2052 D=0.8618 div=0.4785 | FID=172.49
  [ckpt] dcgan_v3__FULL_dcgan_v3_nz100_ngf64__ep100.pt
[score] real=1000 gen=1024


KeyboardInterrupt: 

In [ ]:
# latent interpolation

In [ ]:
import os
from pathlib import Path
import torch
import torch.nn as nn
from torchvision import utils

# ── 1. USTAWIENIA I ŚCIEŻKI ────────────────────────────────────────────
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NZ_V3         = 100  # rozmiar wektora latentnego
NGF_V3        = 64   # rozmiar bazowy filtrów generatora
DRIVE_DIR     = "/content/drive/MyDrive/diffusion_cats"
FINAL_V3_BASE = Path(DRIVE_DIR) / 'final_dcgan_v3'

# Nowe foldery na selekcję danych
POOL_DIR      = FINAL_V3_BASE / 'selection_pool'
INTERP_V3     = FINAL_V3_BASE / 'interpolation_selected'

POOL_DIR.mkdir(parents=True, exist_ok=True)
INTERP_V3.mkdir(parents=True, exist_ok=True)

PATH_TO_WEIGHTS = Path(DRIVE_DIR) / 'dcgan_v3__FULL_dcgan_v3_nz100_ngf64__ep580.pt'

# ── 2. ARCHITEKTURA GENERATORA ────────────────────────────────────────
class DCGeneratorV3(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(NZ_V3, NGF_V3*8, 4, 1, 0),
            nn.BatchNorm2d(NGF_V3*8),
            nn.ReLU(),
            nn.ConvTranspose2d(NGF_V3*8, NGF_V3*4, 4, 2, 1),
            nn.BatchNorm2d(NGF_V3*4),
            nn.ReLU(),
            nn.ConvTranspose2d(NGF_V3*4, NGF_V3*2, 4, 2, 1),
            nn.BatchNorm2d(NGF_V3*2),
            nn.ReLU(),
            nn.ConvTranspose2d(NGF_V3*2, NGF_V3, 4, 2, 1),
            nn.BatchNorm2d(NGF_V3),
            nn.ReLU(),
            nn.ConvTranspose2d(NGF_V3, 3, 4, 2, 1),
            nn.Tanh(),
        )
    def forward(self, z):
        return self.net(z.view(-1, NZ_V3, 1, 1))

# Inicjalizacja i wczytanie wag
G_v3 = DCGeneratorV3().to(DEVICE)
if not PATH_TO_WEIGHTS.exists():
    raise FileNotFoundError(f"Brak wag pod ścieżką: {PATH_TO_WEIGHTS}")
checkpoint = torch.load(PATH_TO_WEIGHTS, map_location=DEVICE)
G_v3.load_state_dict(checkpoint['G'] if (isinstance(checkpoint, dict) and 'G' in checkpoint) else (checkpoint['state_dict'] if (isinstance(checkpoint, dict) and 'state_dict' in checkpoint) else checkpoint))
G_v3.eval()


# ── KROK 1: GENEROWANIE PULI KOTÓW DO WYBORU ───────────────────────────
def generate_selection_pool(num_images=50):
    """
    Generuje określoną liczbę losowych zdjęć do późniejszego wyboru.
    """
    print(f"\n[Krok 1] Generowanie puli {num_images} obrazów do wyboru...")
    torch.manual_seed(42)

    with torch.no_grad():
        for idx in range(1, num_images + 1):
            z = torch.randn(1, NZ_V3)
            torch.save(z, POOL_DIR / f'latent_{idx:03d}.pt')

            img = G_v3(z.to(DEVICE)).cpu().clamp(-1, 1)
            utils.save_image((img + 1) / 2, POOL_DIR / f'cat_{idx:03d}.png')

    print(f"[Gotowe] Przejdź do folderu:\n{POOL_DIR.absolute()}\ni wybierz najładniejsze koty!")


# ── KROK 2: INTERPOLACJA JEDNEGO RAPORTU (DWA WIERSZE) ──────────────────
def generate_single_two_row_report(selected_pairs, steps=10):
    """
    Bierze dokładnie dwie pierwsze pary z listy i generuje z nich
    jeden wspólny raport składający się z dwóch wierszy.
    """
    print(f"\n[Krok 2] Rozpoczynam interpolację liniową dla 2 wybranych par...")
    alphas = [i / (steps - 1) for i in range(steps)]

    # Wybieramy tylko 2 pierwsze pary z przekazanej listy
    chosen_pairs = selected_pairs[:2]

    if len(chosen_pairs) < 2:
        raise ValueError("Lista 'moje_pary' musi zawierać co najmniej 2 pary, aby wygenerować dwuwierszowy raport!")

    report_imgs = []

    with torch.no_grad():
        for id_start, id_end in chosen_pairs:
            print(f"  -> Interpolacja: Kot {id_start:03d} ===> Kot {id_end:03d}")

            path_z1 = POOL_DIR / f'latent_{id_start:03d}.pt'
            path_z2 = POOL_DIR / f'latent_{id_end:03d}.pt'

            if not path_z1.exists() or not path_z2.exists():
                raise FileNotFoundError(f"Nie znaleziono plików latent dla kota {id_start} lub {id_end} w puli selection_pool!")

            z1 = torch.load(path_z1, map_location=DEVICE)
            z2 = torch.load(path_z2, map_location=DEVICE)

            # Generowanie paska LERP
            imgs_lerp = []
            for alpha in alphas:
                z_lerp = (1.0 - alpha) * z1 + alpha * z2
                img = G_v3(z_lerp).cpu()
                imgs_lerp.append(img)

            report_imgs.append(torch.cat(imgs_lerp))

        # Składanie 2 wierszy w jedną siatkę
        final_tensor = torch.cat(report_imgs).clamp(-1, 1)
        combined_grid = utils.make_grid(
            final_tensor,
            nrow=steps,      # 10 zdjęć w wierszu
            padding=6,
            normalize=True,
            pad_value=1.0    # białe tło separatorów
        )

        output_path = INTERP_V3 / 'selected_lerp_final_report.png'
        utils.save_image(combined_grid, output_path)
        print(f"\n[Sukces] Wygenerowano pojedynczy dwuwierszowy raport pod nazwą: {output_path.name}")


# ── 6. WYWOŁANIE ───────────────────────────────────────────────────────

# --- ETAP 1 ---
# (Możesz zakomentować po wygenerowaniu bazy obrazów)
# generate_selection_pool(num_images=60)


# --- ETAP 2 ---
# Podaj pary dla dwóch wierszy raportu (kod weźmie tylko indeksy [0] oraz [1]):
moje_pary = [
    (21, 37),   # Wiersz 1: interpolacja od kota 021 do 037
    (43, 47),   # Wiersz 2: interpolacja od kota 043 do 047
]

generate_single_two_row_report(moje_pary, steps=10)


[Krok 2] Rozpoczynam interpolację liniową dla 2 wybranych par...
  -> Interpolacja: Kot 021 ===> Kot 037
  -> Interpolacja: Kot 043 ===> Kot 047

[Sukces] Wygenerowano pojedynczy dwuwierszowy raport pod nazwą: selected_lerp_final_report.png


In [ ]:
# creation of images

In [ ]:
import os
from pathlib import Path
import torch
import torch.nn as nn
from torchvision import utils

# ── 1. USTAWIENIA, ŚCIEŻKI I HIPERPARAMETRY ────────────────────────────
DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NZ_V3           = 100         # Rozmiar wektora ukrytego (zgodnie z treningiem)
NGF_V3          = 64          # Rozmiar bazowy filtrów generatora

# Główny folder projektu
DRIVE_DIR       = "/content/drive/MyDrive/diffusion_cats"

# Wybierz rozmiar siatki:
GRID_SIZE       = 4           # 4 oznacza siatkę 4x4 (16 obrazków)
N_IMGS          = GRID_SIZE * GRID_SIZE
NUM_GRIDS       = 10          # Liczba niezależnych siatek do wygenerowania

# Dokładna ścieżka do wskazanego pliku wag:
PATH_TO_WEIGHTS = Path(DRIVE_DIR) / 'dcgan_v3__FULL_dcgan_v3_nz100_ngf64__ep580.pt'
OUTPUT_DIR      = Path(DRIVE_DIR)

# ── 2. DEFINICJA ARCHITEKTURY GENERATORA ──────────────────────────────
class DCGeneratorV3(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(NZ_V3, NGF_V3*8, 4, 1, 0),
            nn.BatchNorm2d(NGF_V3*8),
            nn.ReLU(),

            nn.ConvTranspose2d(NGF_V3*8, NGF_V3*4, 4, 2, 1),
            nn.BatchNorm2d(NGF_V3*4),
            nn.ReLU(),

            nn.ConvTranspose2d(NGF_V3*4, NGF_V3*2, 4, 2, 1),
            nn.BatchNorm2d(NGF_V3*2),
            nn.ReLU(),

            nn.ConvTranspose2d(NGF_V3*2, NGF_V3, 4, 2, 1),
            nn.BatchNorm2d(NGF_V3),
            nn.ReLU(),

            nn.ConvTranspose2d(NGF_V3, 3, 4, 2, 1),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z.view(-1, NZ_V3, 1, 1))

# ── 3. INICJALIZACJA I MECHANIZM WCZYTYWANIA WAG ──────────────────────
print(f"Inicjalizacja generatora na urządzeniu: {DEVICE}")
G_v3 = DCGeneratorV3().to(DEVICE)

if not PATH_TO_WEIGHTS.exists():
    raise FileNotFoundError(f"Nie znaleziono pliku wag pod ścieżką: {PATH_TO_WEIGHTS.absolute()}")

print(f"Wczytywanie wag z pliku: {PATH_TO_WEIGHTS}...")
checkpoint = torch.load(PATH_TO_WEIGHTS, map_location=DEVICE)

# Inteligentne wyciąganie wag w zależności od zapisanego formatu:
if isinstance(checkpoint, dict) and 'G' in checkpoint:
    G_v3.load_state_dict(checkpoint['G'])
    print("[Sukces] Wczytano stan generatora ze słownika checkpointu.")
elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
    G_v3.load_state_dict(checkpoint['state_dict'])
    print("[Sukces] Wczytano stan ze słownika 'state_dict'.")
else:
    G_v3.load_state_dict(checkpoint)
    print("[Sukces] Wczytano bezpośredni state_dict wag.")

# Przełączenie w tryb ewaluacji
G_v3.eval()

# ── 4. GENEROWANIE I UKŁADANIE REPECENTACYJNYCH SIATEK ───────────────────
print(f"\nRozpoczynam generowanie {NUM_GRIDS} różnych siatek po {N_IMGS} obrazów...")

with torch.no_grad():
    for i in range(1, NUM_GRIDS + 1):
        # Ustawiamy unikalny seed dla każdej iteracji (np. 1000, 1001, 1002...)
        # Dzięki temu każda siatka będzie inna, ale cały proces pozostanie reprodukowalny.
        torch.manual_seed(1000 + i)

        # Losowanie nowych wektorów ukrytych dla bieżącej siatki
        z_grid = torch.randn(N_IMGS, NZ_V3, device=DEVICE)
        generated_imgs = G_v3(z_grid).cpu()

        # Składanie siatki z obrazów
        grid = utils.make_grid(
            generated_imgs,
            nrow=GRID_SIZE,
            padding=4,
            normalize=True,
            pad_value=1.0
        )

        # Dynamiczna nazwa pliku z uwzględnieniem numeru siatki (np. _grid01, _grid02)
        output_path = OUTPUT_DIR / f'dcgan_report_grid_ep580_{GRID_SIZE}x{GRID_SIZE}_grid{i:02d}.png'
        utils.save_image(grid, output_path)

        print(f" -> Zapisano siatkę {i}/{NUM_GRIDS} w: {output_path.name}")

print('\n[Koniec] Wszystkie siatki zostały wygenerowane pomyślnie!')

Inicjalizacja generatora na urządzeniu: cuda
Wczytywanie wag z pliku: /content/drive/MyDrive/diffusion_cats/dcgan_v3__FULL_dcgan_v3_nz100_ngf64__ep580.pt...
[Sukces] Wczytano stan generatora ze słownika checkpointu.

Rozpoczynam generowanie 10 różnych siatek po 16 obrazów...
 -> Zapisano siatkę 1/10 w: dcgan_report_grid_ep580_4x4_grid01.png
 -> Zapisano siatkę 2/10 w: dcgan_report_grid_ep580_4x4_grid02.png
 -> Zapisano siatkę 3/10 w: dcgan_report_grid_ep580_4x4_grid03.png
 -> Zapisano siatkę 4/10 w: dcgan_report_grid_ep580_4x4_grid04.png
 -> Zapisano siatkę 5/10 w: dcgan_report_grid_ep580_4x4_grid05.png
 -> Zapisano siatkę 6/10 w: dcgan_report_grid_ep580_4x4_grid06.png
 -> Zapisano siatkę 7/10 w: dcgan_report_grid_ep580_4x4_grid07.png
 -> Zapisano siatkę 8/10 w: dcgan_report_grid_ep580_4x4_grid08.png
 -> Zapisano siatkę 9/10 w: dcgan_report_grid_ep580_4x4_grid09.png
 -> Zapisano siatkę 10/10 w: dcgan_report_grid_ep580_4x4_grid10.png

[Koniec] Wszystkie siatki zostały wygenerowane pomyś